# 01. config: YAML → 검증된 설정 객체

`src/quanttool/config.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `configs/*.yaml`, QuantLib(달력·일수 등 부품), pydantic | `Config` 하나 (관례·지수·커브 레시피·엔진 선택·템플릿·리스크 규칙). 다른 모든 파일이 이것을 받는다 |

**객체 지도** (이 순서로 본다)

1. ① 바탕: 문자열 → QuantLib 매핑 사전
2. `load_config` → `Config` (전체 모양 먼저)
3. ② `Conventions` (관례)
4. ③ `IndexConfig`, `Segment`, `CurveConfig` (시장 설정, 금리)
5. ④ `PricingConfig` + 템플릿
6. ⑤ `Override`, `RiskConfig`
7. ⑥ `Config`: 교차 검사, `build_order`

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from pydantic import ValidationError

sorted(p.name for p in (ROOT / "configs").glob("*.yaml"))

설정 파일 네 개. 관례 파일은 YAML 앵커(`&KRW`, `<<: *KRW`)로 돌려쓰기를 한다.

In [ ]:
print((ROOT / "configs" / "conventions.yaml").read_text())

## ① 매핑 사전
YAML의 문자열을 QuantLib 객체로 바꾸는 표. 여기 없는 이름은 읽는 순간 오류가 난다.

In [ ]:
from quanttool.config import BDCS, CALENDARS, CURRENCIES, DAY_COUNTERS, FREQUENCIES

{
    name: sorted(t)
    for name, t in [
        ("CALENDARS", CALENDARS),
        ("DAY_COUNTERS", DAY_COUNTERS),
        ("FREQUENCIES", FREQUENCIES),
        ("CURRENCIES", CURRENCIES),
        ("BDCS", BDCS),
    ]
}

In [ ]:
CALENDARS["SouthKorea.Settlement"]().name(), DAY_COUNTERS["Actual365Fixed"]().name()

## `load_config` → `Config`
입력: configs 폴더 경로. 출력: `Config`. 필드마다 무엇이 들어 있는지 먼저 본다.

In [ ]:
from quanttool.config import load_config

cfg = load_config(ROOT / "configs")
pd.Series(
    {
        f: type(getattr(cfg, f)).__name__
        + (f" {sorted(getattr(cfg, f))}" if isinstance(getattr(cfg, f), dict) else "")
        for f in type(cfg).model_fields
    }
)

## ② `Conventions`: 날짜 규칙
필드는 문자열, `ql_*` 메서드가 QuantLib 객체를 돌려준다. KRW_IRS는 앵커로 KRW_OIS를 받고 주기만 다르다.

In [ ]:
pd.DataFrame({n: c.model_dump() for n, c in cfg.conventions.items()})

In [ ]:
c = cfg.conventions["KRW_IRS"]
(
    c.ql_calendar().name(),
    c.ql_day_count().name(),
    c.ql_frequency() == ql.Quarterly,
    c.ql_bdc() == ql.ModifiedFollowing,
)

검증: 지원하지 않는 값은 만들 때 오류. 만든 뒤에는 고칠 수 없고(frozen), 바꾸려면 `model_copy`.

In [ ]:
from quanttool.config import Conventions

try:
    Conventions(**{**c.model_dump(), "calendar": "Mars"})
except ValidationError as e:
    print(str(e).splitlines()[2])
try:
    c.frequency = "Annual"
except ValidationError as e:
    print("frozen:", str(e).splitlines()[2])
c.model_copy(update={"frequency": "Annual"}).frequency

## ③ 시장 설정 (금리)
### `IndexConfig`: 지수 레시피 → `make(name, handle)`
handle 없이 만들면 커브 미연결(부트스트랩 헬퍼용), 주면 그 커브로 예측한다.

In [ ]:
ic = cfg.indices["CD91"]
display(ic)
unlinked = ic.make("CD91")
linked = ic.make(
    "CD91",
    ql.YieldTermStructureHandle(ql.FlatForward(ql.Date(23, 10, 2026), 0.03, ql.Actual365Fixed())),
)


def is_linked(index):
    try:
        index.forwardingTermStructure().referenceDate()
        return True
    except RuntimeError:  # 빈 Handle
        return False


(
    type(unlinked).__name__,
    unlinked.tenor(),
    unlinked.fixingDays(),
    is_linked(unlinked),
    is_linked(linked),
)

### `Segment`와 `CurveConfig`: 커브 레시피
구간 = (헬퍼 종류, 만기들, 시세 키 틀, 지수). 커브 = 구간들 + 관례 이름 + 할인 커브 이름 + 보간 + 요인 격자.

In [ ]:
cc = cfg.curves["CD91"]
display(pd.DataFrame([s.model_dump() for s in cc.segments]))
{k: v for k, v in cc.model_dump().items() if k != "segments"}

`key`: 구간 + 만기 → 시세 키. `dependencies`: 먼저 만들어야 하는 다른 커브(할인 커브 + 지수의 예측 커브, 자기 자신 제외).

In [ ]:
(
    cc.key(cc.segments[1], "5Y"),
    cc.quote_keys()[:3],
    {n: c.dependencies(cfg.indices) for n, c in cfg.curves.items()},
)

## ④ `PricingConfig`와 상품 템플릿 (products.yaml)

In [ ]:
display(cfg.pricing)
cfg.products

## ⑤ `Override`, `RiskConfig`: 그릭 정의 (risk.yaml)

In [ ]:
from quanttool.config import Override

display(cfg.risk)
cfg.risk.model_copy(
    update={"overrides": {"Bond": Override(shift_size=0.001, scheme="Central")}}
).overrides

## ⑥ `Config`: 교차 검사와 `build_order`
클래스 하나로는 못 하는 "이름이 실제로 있나" 검사를 `model_post_init`이 한다.

In [ ]:
from quanttool.config import Config

broken = {**cfg.curves, "CD91": cfg.curves["CD91"].model_copy(update={"discount_curve": "NOPE"})}
try:
    Config(conventions=cfg.conventions, curves=broken, indices=cfg.indices)
except ValidationError as e:
    print(str(e).splitlines()[1])

`build_order`: 의존 먼저 순서(위상 정렬). 순환이면 오류.

In [ ]:
print(cfg.build_order())
cyc = {**cfg.curves, "KOFR": cfg.curves["KOFR"].model_copy(update={"discount_curve": "CD91"})}
try:
    Config(conventions=cfg.conventions, curves=cyc, indices=cfg.indices).build_order()
except ValueError as e:
    print(e)

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `load_config(root)` | configs 폴더 | `Config` | 항상 처음 |
| `Conventions.ql_*()` | — | QuantLib 달력·일수·주기 | 커브 헬퍼·상품 조립 안에서 |
| `IndexConfig.make(name, handle)` | 이름, 예측 커브 Handle | `ql.IborIndex`/`OvernightIndex` | 헬퍼, 시장 지수 |
| `CurveConfig.key / dependencies` | 구간·만기 / 지수 레시피 | 시세 키 / 커브 이름 집합 | 커브 조립, 순서 계산 |
| `Config.build_order()` | — | 커브 이름 리스트 | `build_market` |
| `model_copy(update=…)` | 바꿀 필드 | 새 설정 객체 | 실험용 변형 |

## 내 메모

- 